In [1]:
import os
import sys
from openai import OpenAI
from pathlib import Path

assert os.environ.get("OPENAI_API_KEY") # "Set OPENAI_API_KEY before running this notebook"
#os.environ["OPENAI_API_URL"] = "https://openai.vocareum.com/v1"
#os.environ["OPENAI_API_KEY"] = "voc-204283627021403739729016a89b188c0a8a4.37364158"

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from src.rag_settings import Settings, RunSummary

_client = OpenAI()

_rag_settings = Settings()
_rag_settings.GENERATE_ANSWER_RAG = False

# This moves up one level from the notebook's location to find the root folder
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.rag_pipeline import (
    chunk_documents,
    chunk_text,
    build_index,
    embed_batch,
    retrieve,
    ask_rag,
    cost_usd,
    DEFAULT_SYSTEM,
)

from src.rag_db_dml import (
    create_collection, 
    upsert_collection, 
    retrive_from_collection
    )

Connected to Qdrant at https://91ecc16f-269a-4046-b44c-5a190057...
Existing collections: ['Book_DALE_CARN', 'wk07_day1_animals', 'wk09_day1_acme', 'wk09_day2_acme']

[] (empty), — this is a fresh cluster.


In [2]:
QUERY01 = "Who were the publishers of the book?" 
QUERY02 =  "when did the Course originate?" 
qry = []
qry.append(QUERY01)
qry.append(QUERY02)
q_vec = []
q_vec = embed_batch(qry)
##q_vec

In [3]:
coll_name = "Book_DALE_CARN"

top5_vec = await retrive_from_collection(q_vec, coll_name, 5)

In [4]:
top5_vec

[ScoredPoint(id=65, version=1, score=0.402613, payload={'source_id': 'HtoWF_02WhythisBook.txt', 'chunk_id': 'HtoWF_02WhythisBook.txt#0', 'text': 'Here is the cleanly formatted version of the introduction using Markdown headings, bullet points, blockquotes, and bold key terms to make it scannable while preserving Dale Carnegie\'s exact wording.\n\nHow This Book Was Written—And Why  \nby Dale Carnegie  \n\nDuring the first thirty-five years of the twentieth century, the publishing houses of America printed more than a fifth of a million different books. Most of them were deadly dull, and many were financial failures. "Many," did I say? The president of one of the largest publishing houses in the world confessed to me that his company, after seventy-five years of publishing experience, still lost money on seven out of every eight books it published.  \n\nWhy, then, did I have the temerity to write another book? And, after I had written it, why'}, vector=None, shard_key=None, order_value=N

In [5]:
print(f"Query: {QUERY01}\n\n====================================\n")
# List comprehension to get structured dictionaries
parsed_results = [
    {
        "chunk_id": pt.payload.get("chunk_id"),
        "source_id": pt.payload.get("source_id"),
        "text": pt.payload.get("text"),
        "score": pt.score,
    }
    for pt in top5_vec
]

# Print extracted values
#for item in parsed_results:
#    print(f"[{item['score']:.4f}] {item['chunk_id']}:\n{item['text']}\n")

Query: Who were the publishers of the book?




In [6]:
parsed_results

[{'chunk_id': 'HtoWF_02WhythisBook.txt#0',
  'source_id': 'HtoWF_02WhythisBook.txt',
  'text': 'Here is the cleanly formatted version of the introduction using Markdown headings, bullet points, blockquotes, and bold key terms to make it scannable while preserving Dale Carnegie\'s exact wording.\n\nHow This Book Was Written—And Why  \nby Dale Carnegie  \n\nDuring the first thirty-five years of the twentieth century, the publishing houses of America printed more than a fifth of a million different books. Most of them were deadly dull, and many were financial failures. "Many," did I say? The president of one of the largest publishing houses in the world confessed to me that his company, after seventy-five years of publishing experience, still lost money on seven out of every eight books it published.  \n\nWhy, then, did I have the temerity to write another book? And, after I had written it, why',
  'score': 0.402613},
 {'chunk_id': 'HtoWF_01Preface.txt#0',
  'source_id': 'HtoWF_01Preface.

In [7]:
import asyncio

SYSTEM = (
    "You are a helpful assistant. Answer the user's question using ONLY the "
    "provided context. If the context does not contain the answer, say so "
    "plainly. Cite the source id in square brackets after any fact you use."
)


tasks = [ask_rag(ques, parsed_results, k=5) for ques in qry]
responses = await asyncio.gather(*tasks)


In [8]:
responses[0]

{'question': 'Who were the publishers of the book?',
 'answer': '',
 'sources': ['HtoWF_02WhythisBook.txt#0',
  'HtoWF_01Preface.txt#0',
  'HtoWF_02WhythisBook.txt#1',
  'HtoWF_02WhythisBook.txt#9',
  'HtoWF_01Preface.txt#3'],
 'tokens_in': 0,
 'tokens_out': 0,
 'retrieved': [{'chunk_id': 'HtoWF_02WhythisBook.txt#0',
   'source_id': 'HtoWF_02WhythisBook.txt',
   'text': 'Here is the cleanly formatted version of the introduction using Markdown headings, bullet points, blockquotes, and bold key terms to make it scannable while preserving Dale Carnegie\'s exact wording.\n\nHow This Book Was Written—And Why  \nby Dale Carnegie  \n\nDuring the first thirty-five years of the twentieth century, the publishing houses of America printed more than a fifth of a million different books. Most of them were deadly dull, and many were financial failures. "Many," did I say? The president of one of the largest publishing houses in the world confessed to me that his company, after seventy-five years of p

In [9]:
import json
from pathlib import Path

# 1. Define the folder path
folder_path = Path("../docs/Dale_Carnigale")

print(f"Checking directory: {folder_path.resolve()}")
print(f"Does folder exist?: {folder_path.exists()}\n")

if folder_path.exists():
    print("Files found in this folder:")
    

golden = {
    row['GoldID']: row 
    for row in (json.loads(line) for line in (folder_path / 'Golden_set_v1_2.jsonl').read_text().splitlines() if line.strip())}

golden

Checking directory: /voc/work/IITM-AI-RAG/rag/docs/Dale_Carnigale
Does folder exist?: True

Files found in this folder:


{'GID0039': {'DocID': 'HtoWF_04_Part01_01.txt',
  'GoldID': 'GID0039',
  'Sno': 1,
  'Question': 'What quote from Benjamin Franklin summarizes his secret to handling people diplomatically?',
  'Answer': '"I will speak ill of no man... and speak all the good I know of everybody."',
  'Citation': 'Doc-HtwWf_04_Part01_01|Para: Paragraph 30',
  'Answer_Type': 'Easy',
  'chunk_id': 'HtoWF_04_Part01_01.txt#15'},
 'GID0089': {'DocID': 'HtoWF_04_Part01_02.txt',
  'GoldID': 'GID0089',
  'Sno': 2,
  'Question': 'Analyze General Alvaro Obregonâ€™s warning: "Be afraid of the friends who flatter you." Why is flattery dangerous to the recipient?',
  'Answer': " Flattery is counterfeit praise designed to manipulate the listener for the flatterer's selfish ends; accepting it creates false pride and leaves one vulnerable to deception by insincere allies.",
  'Citation': 'Doc-HtwWf_04_Part01_02|Para: Paragraphs 42 & 44',
  'Answer_Type': 'Hard',
  'chunk_id': 'HtoWF_04_Part01_02.txt#13'},
 'GID0136': {'

In [17]:
import asyncio

##Evaluation pipeline
responses = []

tasks = []
k = 3

# 1. Batch embed all queries in a single network call
all_questions = [item["Question"] for item in golden.values()]
all_vectors = embed_batch(all_questions)

async def process_goldenset(qry_vec, coll_name, k):
    ##qry = [item["Question"]]

    # 1. Embed (use asyncio.to_thread if embed_batch is synchronous/blocking)
    ##query_vec = embed_batch(qry)

    # 2. Retrieve asynchronously
    top5_vec = await retrive_from_collection(qry_vec, coll_name, k)

    # 3. Parse
    parsed_results = [
        {
            "chunk_id": pt.payload.get("chunk_id"),
            "source_id": pt.payload.get("source_id"),
            "text": pt.payload.get("text"),
            "score": pt.score,
        }
        for pt in top5_vec
    ]

    # 4. Generate RAG response
    return await ask_rag(qry_vec, parsed_results, k)

# Main loop
tasks = [process_goldenset(qry_vec, coll_name, k) for qry_vec in all_vectors]
responses = await asyncio.gather(*tasks)


In [18]:
len(responses)

24

In [19]:
for item, resp in zip(golden.values(), responses):
    hit_sources = resp.get("sources", [])
    resp["hit_rate"] = 1 if item["chunk_id"] in hit_sources else 0
    resp["cost"] = cost_usd(resp["tokens_in"], resp["tokens_out"])
    resp["chunk_id"] = item["chunk_id"]
    resp["query"] = item["Question"]
    #result.append(resp)


In [20]:
for result in responses:
    print(f"\n Q: {result['query']}")
    print(f"A: {result['answer']}")
    print(f"Hit rate: {result['hit_rate']}\nGolden Chunk_id: {result['chunk_id']} <==> \nSources retrieved: {result['sources']}")
    print(f"Prompt tokens: {result['tokens_in']} | Completion tokens: {result['tokens_out']} || Latency: {result['latency_s']}; Cost in USD:{result['cost']} ")


 Q: What quote from Benjamin Franklin summarizes his secret to handling people diplomatically?
A: 
Hit rate: 0
Golden Chunk_id: HtoWF_04_Part01_01.txt#15 <==> 
Sources retrieved: ['HtoWF_04_Part01_03.txt#13', 'HtoWF_04_Part01_01.txt#11', 'HtoWF_04_Part01_02.txt#13']
Prompt tokens: 0 | Completion tokens: 0 || Latency: 6.589980330318213e-07; Cost in USD:0.0 

 Q: Analyze General Alvaro Obregonâ€™s warning: "Be afraid of the friends who flatter you." Why is flattery dangerous to the recipient?
A: 
Hit rate: 1
Golden Chunk_id: HtoWF_04_Part01_02.txt#13 <==> 
Sources retrieved: ['HtoWF_04_Part01_02.txt#13', 'HtoWF_04_Part01_02.txt#12', 'HtoWF_04_Part01_01.txt#6']
Prompt tokens: 0 | Completion tokens: 0 || Latency: 7.00994860380888e-07; Cost in USD:0.0 

 Q: Why did White House head usher Ike Hoover say Rooseveltâ€™s visit was "the only happy day we had in nearly two years"?
A: 
Hit rate: 1
Golden Chunk_id: HtoWF_04_Part02_01.txt#11 <==> 
Sources retrieved: ['HtoWF_04_Part02_01.txt#12', 'Ht

In [21]:
Total = 0
Hit_rate = 0
Latency = 0
total_cost_usd = 0
for result in responses:
    #print(f"\n Q: {result['query']}:  A: {result['answer']}")
    #print(f"Hit rate: {result['hit_rate']}\nGolden Chunk_id: {result['chunk_id']} <==> \nSources retrieved: {result['sources']}")
    Total = Total + 1
    Hit_rate = Hit_rate + result['hit_rate']
    Latency = Latency + result['latency_s']
    total_cost_usd = total_cost_usd + result['cost']

print(f"Hit rate: {Hit_rate}/{Total} | Latency: {Latency} | Total Cost: ${total_cost_usd}")


Hit rate: 20/24 | Latency: 8.4270432125777e-06 | Total Cost: $0.0


In [22]:
outpath = f"./response_output.txt"
with open(outpath, "w", encoding="utf-8") as file:
    for item in responses:
        file.write(f"{item}\n")